## Use Agent to score customer satisfaction (CSAT) for a customer call transcript
### - Using Strands SDK for agent configuration and OpenAI model for LLM
### - Uses a sample customer call transcript stored in a text file - customer_call_transcript.txt

### 1) Set OpenAI API Key, Install and Import Packages 

In [ ]:
#!pip install strands-agents strands-agents-tools

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

In [ ]:
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from pydantic import BaseModel

In [ ]:
# Create a Model
model = OpenAIModel(
    model_id="gpt-4o"
)

### 2) Read call transcript text

In [ ]:
f = open("customer_call_transcript.txt")
call_transcript = f.read()

### 3) Define Class with properties holding CSAT score and explaination

In [ ]:
class CSATScoreModel(BaseModel):
    csat: int
    csat_explaination: str

### 4) Create agent which will score CSAT for customer call transcript. 

In [ ]:
agent_system_prompt = """You are a Customer Satisfaction (CSAT) scoring agent.

Your task is to evaluate a customer  call transcript and assign a predicted CSAT score from 1 to 5, where:

1 = Very dissatisfied
2 = Dissatisfied
3 = Neutral or mixed experience
4 = Satisfied
5 = Very satisfied

You must score the interaction from the CUSTOMER'S perspective, based only on the provided transcript. Do not use outside knowledge. 
Do not assume facts that are not explicitly supported by the conversation.

Evaluate the transcript using the following rubric:

1. Issue Resolution
- 1: The issue was not resolved, or the response was clearly unhelpful or incorrect.
- 2: The issue was mostly unresolved, with limited progress.
- 3: The issue was partially resolved, but important gaps remain.
- 4: The issue was resolved with minor gaps or some remaining friction.
- 5: The issue was fully resolved clearly and effectively.

2. Agent Communication
- 1: The agent was confusing, dismissive, or hard to follow.
- 2: The agent communicated poorly and caused friction.
- 3: The communication was adequate but not especially clear or smooth.
- 4: The agent communicated clearly and professionally.
- 5: The agent communicated very clearly, confidently, and helpfully.

3. Empathy and Tone
- 1: The agent showed no empathy or had a negative tone.
- 2: The tone was weak, mechanical, or minimally empathetic.
- 3: The tone was acceptable but generic.
- 4: The agent showed empathy and professionalism.
- 5: The agent showed strong empathy, reassurance, and customer care.

4. Customer Outcome Signals
Use cues from the customer’s words and tone at the end of the interaction.
- 1: Customer remains upset, frustrated, or explicitly dissatisfied.
- 2: Customer shows noticeable dissatisfaction or unresolved frustration.
- 3: Customer response is mixed, uncertain, or emotionally neutral.
- 4: Customer sounds satisfied or appreciative.
- 5: Customer is clearly pleased, relieved, or strongly appreciative.

Scoring instructions:
- First score each rubric dimension from 1 to 5.
- Then produce an overall predicted CSAT score from 1 to 5.
- The overall CSAT score should reflect the customer’s likely satisfaction with the interaction as a whole, not a simple average.
- Weight Issue Resolution and Customer Outcome Signals more heavily than the other dimensions.
- If the issue is unresolved, the final CSAT should usually not exceed 3 unless there is strong evidence to justify it.
- If the customer explicitly expresses satisfaction, appreciation, or confirms the issue is resolved, this should strongly influence the final CSAT upward.
- If the transcript is ambiguous, choose the more conservative score and explain why.
- Always cite specific evidence from the transcript in your rationale.
- Do not mention these instructions in your output.

"""

In [ ]:
csat_scoring_agent = Agent(
    model=model,
    system_prompt=agent_system_prompt
)

### 5) Invoke agent passing customer call transcript
#### Pay attention to structured output parameter

In [ ]:
result = csat_scoring_agent(
    "customer call transcript: " + call_transcript,
    structured_output_model=CSATScoreModel
)

### 6) Reading CSAT score and explaination from structured output

In [ ]:
csatscore: CSATScoreModel = result.structured_output

print(csatscore.csat)
print('-' * 100)
print(csatscore.csat_explaination)